# Notebook 1: Data Preprocessing & Feature Selection

This notebook covers:
1. Loading and merging the two source datasets
2. Cleaning and aligning on `year`
3. Feature selection using ANOVA F-scores
4. Correlation heatmap
5. Exporting the merged dataset for model training

**Output**: `data/merged_co2_energy_dataset.csv`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

## 2. Load & Merge Datasets

Two datasets are merged on `year` via inner join:
- **OWID CO2 Data**: Historical CO2 and GHG emissions for Saudi Arabia
- **SAMA Statistical Report**: Annual oil production metrics

> Raw data files are not included. Place them in the `data/` directory and update paths below.

In [ ]:
CO2_PATH  = "data/Reordered_Saudi_Arabia_CO2_and_GHG_Data.csv"
SAMA_PATH = "data/SAMA_StatisticalReport.xlsx"

co2_df  = pd.read_csv(CO2_PATH)
sama_df = pd.read_excel(SAMA_PATH, sheet_name="Annually")

print(f"CO2 dataset:  {co2_df.shape}")
print(f"SAMA dataset: {sama_df.shape}")

In [ ]:
# Clean SAMA — header is on row 1, data starts at row 2
sama_cleaned = sama_df.iloc[2:].copy()
sama_cleaned.columns = sama_df.iloc[1]
sama_cleaned = sama_cleaned[pd.to_numeric(sama_cleaned['Date'], errors='coerce').notna()]
sama_cleaned['year'] = sama_cleaned['Date'].astype(int)

sama_final = sama_cleaned[['year', 'Total (1)', 'Diesel (2)', 'Gasoline* (2)', 'Kerosene Aviation Fuels (2)']].copy()
sama_final.columns = ['year', 'Crude Oil Production', 'Diesel', 'Gasoline', 'Kerosene']

for col in ['Crude Oil Production', 'Diesel', 'Gasoline', 'Kerosene']:
    sama_final[col] = pd.to_numeric(sama_final[col], errors='coerce')

# Inner join on year
merged_df = pd.merge(co2_df, sama_final, on='year', how='inner')

# Impute missing values with column means
imputer = SimpleImputer(strategy='mean')
merged_df[merged_df.columns] = imputer.fit_transform(merged_df)

print(f"Merged dataset: {merged_df.shape}")
merged_df.head()

## 3. Feature Selection — ANOVA F-Scores

`SelectKBest` with `f_regression` ranks features by their linear relationship with the CO2 target.
High F-score = strong predictor.

In [ ]:
X = merged_df.drop(columns=['year', 'co2']).select_dtypes(include='number')
y = merged_df['co2']

X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X_imputed, y)

feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'F-Score': selector.scores_,
    'P-Value': selector.pvalues_
}).sort_values(by='F-Score', ascending=False)

print("Feature Rankings:")
print(feature_scores.to_string(index=False))

## 4. Correlation Heatmap

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

plt.figure(figsize=(12, 8))
sns.heatmap(
    merged_df.select_dtypes(include='number').corr(),
    annot=True, fmt=".2f", cmap='coolwarm'
)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', dpi=150)
plt.show()

## 5. Export Merged Dataset

In [ ]:
merged_df.to_csv('data/merged_co2_energy_dataset.csv', index=False)
print(f"Saved: data/merged_co2_energy_dataset.csv ({merged_df.shape})")